In [0]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.functions import regexp_replace
from pyspark.sql.window import Window
from pyspark.sql.functions import col
from pyspark.sql.types import StringType
from pyspark.sql.functions import left

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

volume_path = "/Volumes/data_warehouse/bndes/files_ingest"

files = dbutils.fs.ls(volume_path)
latest_file = sorted(files, key=lambda f: f.modificationTime)[-1]

print(f"Latest file name: {latest_file.name}")

if latest_file.name =="operacoes-financiamento-operacoes-nao-auto_carga.csv":
    print("Arquivo ja carregado")
    dbutils.jobs.taskValues.set(key = 'run_tsk', value = False)

elif "operacoes-financiamento-operacoes-nao-auto_20" in latest_file.name :
    dbutils.fs.cp(
    "/Volumes/data_warehouse/bndes/files_ingest/"+latest_file.name,
    "/Volumes/data_warehouse/bndes/files_ingest/operacoes-financiamento-operacoes-nao-auto_carga.csv")
    
    entry_file_raw = "/Volumes/data_warehouse/bndes/files_ingest/operacoes-financiamento-operacoes-nao-auto_carga.csv"
    file_data_ref = latest_file.name[-10:].replace(".csv", "")

    dbutils.jobs.taskValues.set(key = 'run_tsk', value = True)
    dbutils.jobs.taskValues.set(key = 'entry_file_raw_tsk', value = entry_file_raw)
    dbutils.jobs.taskValues.set(key = 'file_data_ref_tsk', value = file_data_ref)
    print("Arquivo carregado:" +file_data_ref)

else :
    print("Arquivo não encontrado")
    dbutils.jobs.taskValues.set(key = 'run_tsk', value = False)